In [ ]:
import pandas as pd
import numpy as np
import json
import os

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/url_features_extracted1.csv')
print("=== DATASET OVERVIEW ===")
print(f"Shape: {df.shape}")
print(f"\nColumn names:\n{list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nClass distribution:")
print(df['ClassLabel'].value_counts())
print(f"\nClass balance: {df['ClassLabel'].value_counts(normalize=True).round(3)}")

print(f"\nStatistical summary:")


print(f"\nMissing values per column:")
print(df.isnull().sum())

print(f"\nDuplicate rows: {df.duplicated().sum()}")

=== DATASET OVERVIEW ===
Shape: (101219, 18)

Column names:
['URL', 'url_length', 'has_ip_address', 'dot_count', 'https_flag', 'url_entropy', 'token_count', 'subdomain_count', 'query_param_count', 'tld_length', 'path_length', 'has_hyphen_in_domain', 'number_of_digits', 'tld_popularity', 'suspicious_file_extension', 'domain_name_length', 'percentage_numeric_chars', 'ClassLabel']

Data types:
URL                              str
url_length                     int64
has_ip_address                 int64
dot_count                      int64
https_flag                     int64
url_entropy                  float64
token_count                    int64
subdomain_count                int64
query_param_count              int64
tld_length                     int64
path_length                    int64
has_hyphen_in_domain           int64
number_of_digits               int64
tld_popularity                 int64
suspicious_file_extension      int64
domain_name_length             int64
percentage_num

In [2]:
# Check duplicates
print(f"Duplicate URLs: {df['URL'].duplicated().sum()}")
df = df.drop_duplicates(subset=['URL'])
print(f"Shape after removing duplicates: {df.shape}")

Duplicate URLs: 346
Shape after removing duplicates: (100873, 18)


In [3]:
# Quick URL sanity check
print("Sample phishing URLs:")
print(df[df['ClassLabel']==1.0]['URL'].sample(5).values)

print("\nSample legitimate URLs:")
print(df[df['ClassLabel']==0.0]['URL'].sample(5).values)

# Check URL column is actually URLs
has_http = df['URL'].str.startswith(('http://', 'https://')).mean()
print(f"\nURLs starting with http/https: {has_http:.1%}")

Sample phishing URLs:
<ArrowStringArray>
[          'https://www.coare.org', 'https://www.sheetmetalworld.com',
 'https://www.ozfoodhunter.com.au',     'https://www.capitalxtra.com',
          'https://www.albizu.edu']
Length: 5, dtype: str

Sample legitimate URLs:


<ArrowStringArray>
[                                    'http://216.155.92.204:2876/Mozi.m',
                                          'http://223.10.25.138:51002/i',
 'http://nsdm.cumpar-auto-orice-tip.ro/prog/66e877160911d_vnfdewk16.exe',
                                   'http://202.191.123.196:27033/Mozi.m',
                           'grafikicon.com/my-paylink-testpageonly.html']
Length: 5, dtype: str

URLs starting with http/https: 98.7%


In [ ]:
import os

folders = [
    'data/raw', 'data/processed', 'data/logs',
    'notebooks', 'src', 'models', 
    'dashboard', 'tests', 'reports'
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✓ {folder}")

print("\nProject structure verified and ready.")

✓ data/raw
✓ data/processed
✓ data/logs
✓ notebooks
✓ src
✓ models
✓ dashboard
✓ tests
✓ reports

Project structure verified and ready.


In [5]:
# Save a summary report for documentation
report = {
    'total_rows': len(df),
    'total_features': len(df.columns) - 3,  # exclude URL, domain, ClassLabel
    'phishing_count': int((df['ClassLabel']==1.0).sum()),
    'legitimate_count': int((df['ClassLabel']==0.0).sum()),
    'missing_values': int(df.isnull().sum().sum()),
    'duplicate_urls': 0,  # after removal
    'dataset_source': 'LegitPhish - Mendeley Data v1'
}

import json
with open('reports/phase1_data_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("Data quality report saved to reports/phase1_data_report.json")

Data quality report saved to reports/phase1_data_report.json


In [6]:
# Verify label meaning
print("ClassLabel = 1.0 (supposed to be Phishing):")
print(df[df['ClassLabel']==1.0]['URL'].sample(10, random_state=1).values)

print("\nClassLabel = 0.0 (supposed to be Legitimate):")
print(df[df['ClassLabel']==0.0]['URL'].sample(10, random_state=1).values)

print("\nVerify against dataset documentation:")
print("If phishing URLs look legitimate above — labels are SWAPPED")
print("If legitimate URLs look suspicious above — labels are SWAPPED")

ClassLabel = 1.0 (supposed to be Phishing):
<ArrowStringArray>
[   'https://www.fishingheritage.com',            'https://www.skyphone.jp',
        'https://www.reklamy-arek.pl',                'https://www.sgo.org',
         'https://www.knotsindia.com',             'https://www.sportal.de',
             'https://www.opcmia.org',      'https://www.animus-travel.com',
 'https://www.reliablehomeimprov.com',    'https://www.campingroadtrip.com']
Length: 10, dtype: str

ClassLabel = 0.0 (supposed to be Legitimate):
<ArrowStringArray>
[                                   'http://198.98.51.37:27222/s/arm7',
                                        'http://119.116.26.98:43642/i',
                          'http://147.45.44.104/ldms/524f141e189d.exe',
                                    'http://59.91.82.149:37525/bin.sh',
                                     'http://172.232.38.224/ttssgg.sh',
                                   'http://186.97.185.92:31376/Mozi.m',
              'http://males.mug

In [7]:
# Fix swapped labels
df['ClassLabel'] = df['ClassLabel'].map({1.0: 0.0, 0.0: 1.0})
print("Labels corrected")
print(df['ClassLabel'].value_counts())

Labels corrected
ClassLabel
1.0    63492
0.0    37380
Name: count, dtype: int64


In [8]:
# Save cleaned dataset
df.to_csv('../data/raw/url_features_cleaned.csv', index=False)
print(f"Cleaned dataset saved: {df.shape}")
print(f"Duplicates removed: 346")
print(f"Missing ClassLabel removed: 1")

Cleaned dataset saved: (100873, 18)
Duplicates removed: 346
Missing ClassLabel removed: 1
